# CornerScout · 01 Ingesta y trazabilidad

**Responsabilidad única:** restaurar o descargar de forma reanudable el raw de
StatsBomb Open Data para LaLiga 2015/16, sin sobrescribir ningún archivo
existente, y publicar un contrato verificable para el notebook 02.

La descarga está fijada a una revisión del repositorio público. Una ejecución
sobre una copia completa solo valida contenido, cobertura y hashes.

## 0 · Entorno

No se requieren credenciales ni APIs de pago. En Colab, Drive se monta de forma
explícita. No se ocultan warnings del proveedor o de dependencias.

In [1]:
import importlib.util
import subprocess
import sys

PACKAGES = {
    "numpy": "numpy>=1.26,<3",
    "pandas": "pandas>=2.2,<4",
    "matplotlib": "matplotlib>=3.8,<4",
    "pyarrow": "pyarrow>=16",
    "httpx": "httpx>=0.28,<1",
    "tqdm": "tqdm>=4.66",
}
missing = [spec for module, spec in PACKAGES.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Dependencias:", "instaladas las ausentes" if missing else "ya disponibles")

Dependencias: ya disponibles


In [2]:
import csv
import gzip
import hashlib
import io
import json
import os
import platform
import time
from collections import Counter
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import httpx
import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

DATA_DIR = Path(os.environ.get(
    "CORNERSCOUT_DATA_DIR",
    "/content/drive/MyDrive/Corner_Scout/data" if IN_COLAB else "data",
))
RAW_DIR = DATA_DIR / "raw"
EVENTS_DIR = RAW_DIR / "events"
OUT = DATA_DIR / "interim" / "01_ingestion"
for path in (RAW_DIR, EVENTS_DIR, OUT):
    path.mkdir(parents=True, exist_ok=True)

COMPETITION_ID = 11
SEASON_ID = 27
REVISION = "4b73468fc5b0f1950f9f66fada70ad3a4f9327cb"
BASE_URL = f"https://raw.githubusercontent.com/statsbomb/open-data/{REVISION}/data"
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
ENVIRONMENT = {name: version(name) for name in ["pandas", "pyarrow", "httpx", "tqdm"]}
display(pd.Series({"data": str(DATA_DIR), "revision": REVISION, "run_id": RUN_ID,
                   "python": platform.python_version(), **ENVIRONMENT}))

Mounted at /content/drive


,0
data,/content/drive/MyDrive/Corner_Scout/data
revision,4b73468fc5b0f1950f9f66fada70ad3a4f9327cb
run_id,20260922T011346_842854Z
python,3.13.15
pandas,2.2.3
pyarrow,23.0.1
httpx,0.28.1
tqdm,4.67.3


## 1 · Funciones de ingesta inmutable

Cada payload se completa en memoria y se crea con modo exclusivo `xb`. Un
archivo existente nunca se reescribe ni se “repara” silenciosamente.

In [3]:
def sha256_bytes(payload):
    return hashlib.sha256(payload).hexdigest()


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def create_bytes(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("xb") as stream:
        stream.write(payload)


def fetch_json(relative):
    for attempt in range(4):
        try:
            response = httpx.get(f"{BASE_URL}/{relative}", timeout=90, follow_redirects=True)
            response.raise_for_status()
            return response.json()
        except httpx.HTTPError:
            if attempt == 3:
                raise
            time.sleep(2 ** attempt)
    raise RuntimeError("Descarga inaccesible")


def csv_payload(frame):
    return frame.to_csv(index=False).encode("utf-8")


def json_payload(value):
    return json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False).encode("utf-8")


def event_name(value):
    return value.get("name") if isinstance(value, dict) else value


def event_pass_type(record):
    return event_name(record.get("pass_type")) or event_name((record.get("pass") or {}).get("type"))


def inspect_event_file(path):
    rows = corners = 0
    ids = set()
    with gzip.open(path, "rt", encoding="utf-8") as stream:
        for line in stream:
            if not line.strip():
                continue
            record = json.loads(line)
            event_id = record.get("id")
            if not isinstance(event_id, str) or not event_id or event_id in ids:
                raise ValueError(f"IDs ausentes o duplicados en {path.name}")
            ids.add(event_id)
            rows += 1
            corners += int(event_name(record.get("type")) == "Pass"
                           and event_pass_type(record) == "Corner")
    if not rows:
        raise ValueError(f"Archivo vacío: {path}")
    return rows, corners

## 2 · Metadatos de competición y partidos

Si los archivos ya existen se leen y validan. Solo una ruta ausente activa una
descarga pública. Los nombres aprobados dentro de raw no cambian.

In [4]:
competitions_path = RAW_DIR / "competitions.csv"
matches_path = RAW_DIR / "matches_laliga_2015_16.csv"

if not competitions_path.exists():
    create_bytes(competitions_path, csv_payload(pd.DataFrame(fetch_json("competitions.json"))))
if not matches_path.exists():
    matches_records = fetch_json(f"matches/{COMPETITION_ID}/{SEASON_ID}.json")
    create_bytes(matches_path, csv_payload(pd.json_normalize(matches_records, sep="_")))

competitions = pd.read_csv(competitions_path)
matches_raw = pd.read_csv(matches_path)
for side in ("home", "away"):
    alternative = f"{side}_team_{side}_team_name"
    if f"{side}_team" not in matches_raw and alternative in matches_raw:
        matches_raw[f"{side}_team"] = matches_raw[alternative]
required = ["match_id", "match_date", "kick_off", "home_team", "away_team"]
assert set(required) <= set(matches_raw), f"Faltan columnas: {set(required) - set(matches_raw)}"
matches = matches_raw[required].copy()
matches["match_id"] = pd.to_numeric(matches.match_id, errors="raise").astype(int)
assert len(matches) == matches.match_id.nunique() == 380
assert len(set(matches.home_team) | set(matches.away_team)) == 20
display(matches.head())
print(f"Partidos: {len(matches)} · equipos: 20 · {matches.match_date.min()} a {matches.match_date.max()}")

,match_id,match_date,kick_off,home_team,away_team
0,3825562,2015-08-21,20:30:00.000,Málaga,Sevilla
1,3825563,2015-08-22,20:30:00.000,Atlético Madrid,Las Palmas
2,3825564,2015-08-22,18:30:00.000,RC Deportivo La Coruña,Real Sociedad
3,3825565,2015-08-22,18:30:00.000,Espanyol,Getafe
4,3825566,2015-08-22,22:30:00.000,Rayo Vallecano,Valencia


Partidos: 380 · equipos: 20 · 2015-08-21 a 2016-05-15


## 3 · Restauración o descarga de eventos

La ejecución es reanudable. Los archivos existentes se inspeccionan; los
ausentes se descargan y se crean una sola vez.

In [5]:
download_log = []
for match_id in tqdm(matches.match_id, desc="Eventos"):
    target = EVENTS_DIR / f"{match_id}.jsonl.gz"
    status = "existing"
    if not target.exists():
        records = fetch_json(f"events/{match_id}.json")
        text = "".join(json.dumps(row, ensure_ascii=False) + "\n" for row in records)
        create_bytes(target, gzip.compress(text.encode("utf-8"), mtime=0))
        status = "downloaded"
    rows, corners = inspect_event_file(target)
    download_log.append({"match_id": int(match_id), "status": status,
                         "rows": rows, "corners": corners})

ingestion = pd.DataFrame(download_log)
assert len(ingestion) == 380 and ingestion.rows.gt(0).all()
assert set(ingestion.match_id) == set(matches.match_id)
display(ingestion.status.value_counts().rename("partidos"))
print(f"Eventos: {ingestion.rows.sum():,} · córners: {ingestion.corners.sum():,}")

Eventos:   0%|          | 0/380 [00:00<?, ?it/s]

,partidos
status,
existing,380


Eventos: 1,295,354 · córners: 3,841


## 4 · Metadatos raw y contrato derivado

Los metadatos raw solo se crean si faltan. El manifiesto científico vive en
`interim/01_ingestion` y puede regenerarse sin alterar raw.

In [6]:
metadata_path = RAW_DIR / "metadata_ingesta.json"
registry_path = RAW_DIR / "registro_ingesta.csv"
if not metadata_path.exists():
    create_bytes(metadata_path, json_payload({
        "provider": "statsbomb_open_data", "revision": REVISION,
        "exported_at": datetime.now(timezone.utc).isoformat(),
        "format": "provider_nested_jsonl", "competition_id": COMPETITION_ID,
        "season_id": SEASON_ID,
    }))
if not registry_path.exists():
    create_bytes(registry_path, csv_payload(ingestion))

metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
registry = pd.read_csv(registry_path)
assert metadata.get("competition_id") == COMPETITION_ID
assert metadata.get("season_id") == SEASON_ID
assert set(pd.to_numeric(registry.match_id, errors="raise").astype(int)) == set(matches.match_id)

manifest_rows = []
for path in sorted(RAW_DIR.rglob("*")):
    if path.is_file() and path.name != ".gitkeep":
        manifest_rows.append({"relative_path": path.relative_to(RAW_DIR).as_posix(),
                              "sha256": sha256_file(path), "bytes": path.stat().st_size})
manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(OUT / "source_manifest.csv", index=False)
manifest_fingerprint = sha256_bytes(
    json.dumps(manifest_rows, sort_keys=True, separators=(",", ":")).encode("utf-8")
)

contract = {
    "stage": "01_ingestion", "contract_version": "01-ingestion-v1",
    "run_id": RUN_ID, "provider": "statsbomb_open_data", "revision": REVISION,
    "competition_id": COMPETITION_ID, "season_id": SEASON_ID,
    "counts": {"matches": int(len(matches)), "teams": 20,
               "event_files": int(len(list(EVENTS_DIR.glob("*.jsonl.gz")))),
               "events": int(ingestion.rows.sum()), "corners": int(ingestion.corners.sum())},
    "raw_manifest_sha256": manifest_fingerprint,
    "raw_files": {name: sha256_file(RAW_DIR / name) for name in
                  ["competitions.csv", "matches_laliga_2015_16.csv",
                   "metadata_ingesta.json", "registro_ingesta.csv"]},
    "environment": ENVIRONMENT, "python": platform.python_version(),
}
(OUT / "contract.json").write_text(
    json.dumps(contract, ensure_ascii=False, indent=2), encoding="utf-8"
)
display(pd.Series(contract["counts"]))
print("Contrato:", OUT / "contract.json")

,0
matches,380
teams,20
event_files,380
events,1295354
corners,3841


Contrato: /content/drive/MyDrive/Corner_Scout/data/interim/01_ingestion/contract.json


### Cierre

La etapa queda aprobada cuando hay 380 partidos y archivos, los conteos se
reconcilian y el contrato identifica exactamente la copia raw utilizada. Esta
ejecución no modificó ningún archivo raw existente.